In [ ]:
!pip install -q transformers accelerate

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
SPLIT_ID = 0

DATA_DIR          = "/content/drive/MyDrive/data_p4_kpdl"
MODEL_PATH        = "/content/drive/MyDrive/absa_self_train_phase1/asc_teacher_phase1"
CATEGORY_OUTPUT   = "kindle_store"

POLAR_THRESHOLD   = 0.90
NEUTRAL_THRESHOLD = 0.55
BATCH_SIZE        = 256
MAX_LENGTH        = 192
CHUNK_SIZE        = 50_000

print(f"SPLIT_ID : {SPLIT_ID}")
print(f"Thresholds: polar={POLAR_THRESHOLD}, neutral={NEUTRAL_THRESHOLD}")

SPLIT_ID : 0
Thresholds: polar=0.9, neutral=0.55


In [ ]:
import os
import gc
import re
import json
import math
import time
from collections import defaultdict
from contextlib import nullcontext

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SPLITS_DIR = os.path.join(DATA_DIR, "splits")
OUTPUT_DIR = os.path.join(DATA_DIR, "asc_output", f"split_{SPLIT_ID:02d}")
CHUNKS_DIR = os.path.join(OUTPUT_DIR, "chunks")
CKPT_PATH  = os.path.join(OUTPUT_DIR, "checkpoint.json")

assert os.path.exists(SPLITS_DIR), (
    f"Splits dir not found: {SPLITS_DIR}\n"
)
os.makedirs(CHUNKS_DIR, exist_ok=True)

device  = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"

print(f"Device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


In [ ]:
_ws = re.compile(r"\s+")

def clean_text(text):
    if pd.isna(text):
        return ""
    return _ws.sub(" ", str(text)).strip()


def mark_aspect(sentence, aspect):
    pattern = re.compile(re.escape(aspect), flags=re.IGNORECASE)
    marked = pattern.sub(f"[ASP] {aspect} [/ASP]", sentence, count=1)
    if marked == sentence:
        marked = f"{sentence} [ASP] {aspect} [/ASP]"
    return marked


def load_checkpoint():
    if os.path.exists(CKPT_PATH):
        with open(CKPT_PATH) as f:
            return json.load(f)
    return {}


def save_checkpoint(ckpt):
    tmp = CKPT_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump(ckpt, f, indent=2)
    os.replace(tmp, CKPT_PATH)


@torch.no_grad()
def predict_batch(texts, tokenizer, model):
    all_probs = []
    ctx = torch.amp.autocast("cuda", dtype=torch.float16) if use_amp else nullcontext()

    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i : i + BATCH_SIZE]
        enc = tokenizer(
            batch,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_tensors="pt",
        ).to(device)

        with ctx:
            logits = model(**enc).logits

        probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()
        all_probs.append(probs)
        del enc, logits

    return np.concatenate(all_probs, axis=0)


def process_chunk(chunk_df, tokenizer, model):
    sentences   = chunk_df["sentence_text"].values
    aspects_arr = chunk_df["aspects"].values

    row_idxs, flat_sents, flat_asps = [], [], []

    for i in range(len(chunk_df)):
        aspects = aspects_arr[i]
        if not isinstance(aspects, (list, np.ndarray)) or len(aspects) == 0:
            continue
        sent = clean_text(str(sentences[i]))
        for asp in aspects:
            asp_str = clean_text(str(asp))
            if asp_str:
                row_idxs.append(i)
                flat_sents.append(sent)
                flat_asps.append(asp_str)

    if not row_idxs:
        return [], []

    marked = [mark_aspect(s, a) for s, a in zip(flat_sents, flat_asps)]

    lengths  = np.array([len(t) for t in marked])
    sort_idx = np.argsort(lengths)
    sorted_marked = [marked[j] for j in sort_idx]

    sorted_probs = predict_batch(sorted_marked, tokenizer, model)

    all_probs = np.empty_like(sorted_probs)
    all_probs[sort_idx] = sorted_probs

    row_sentiments = defaultdict(list)
    for j, idx in enumerate(row_idxs):
        row_sentiments[idx].append(all_probs[j].tolist())

    high_rows, low_rows = [], []

    for idx, sentiments in row_sentiments.items():
        row = chunk_df.iloc[idx]

        is_high = True
        for probs in sentiments:
            pred_id = int(np.argmax(probs))
            conf = probs[pred_id]
            thr = NEUTRAL_THRESHOLD if pred_id == 1 else POLAR_THRESHOLD
            if conf < thr:
                is_high = False
                break

        aspects_clean = [str(a) for a in row["aspects"] if str(a).strip()]
        base = {
            "parent_asin":   row["parent_asin"],
            "sentence_id":   row["sentence_id"],
            "sentence_text": row["sentence_text"],
            "rating":        row["rating"],
            "category_name": CATEGORY_OUTPUT,
            "aspects":       aspects_clean,
        }

        if is_high:
            base["gate_confidence"] = row["gate_confidence"]
            base["sentiments"]      = sentiments
            high_rows.append(base)
        else:
            low_rows.append(base)

    return high_rows, low_rows


def merge_chunks(label):
    prefix = f"__{label}.parquet"
    files = sorted(f for f in os.listdir(CHUNKS_DIR) if f.endswith(prefix))
    if not files:
        return 0

    out_path = os.path.join(OUTPUT_DIR, f"asc_{label}_{CATEGORY_OUTPUT}.parquet")
    writer = None
    total = 0

    for f in files:
        table = pq.read_table(os.path.join(CHUNKS_DIR, f))
        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema)
        writer.write_table(table)
        total += len(table)
        del table

    if writer:
        writer.close()

    print(f"  {label}: {total:,} rows -> {out_path}")
    return total


print("Functions defined.")

Functions defined.


## Load data + Check checkpoint

In [ ]:
# Load metadata
meta_path = os.path.join(SPLITS_DIR, "meta.json")
with open(meta_path) as f:
    meta = json.load(f)

# Check if already done
ckpt = load_checkpoint()
if ckpt.get("merge_complete"):
    print("Da hoan thanh split nay! Final files da luu.")
    print(f"  High: {ckpt.get('n_high', '?'):,}")
    print(f"  Low : {ckpt.get('n_low', '?'):,}")
else:
    # Read split
    split_path = os.path.join(SPLITS_DIR, f"split_{SPLIT_ID:02d}.parquet")
    print(f"Reading: {split_path}")
    df_full = pd.read_parquet(split_path)
    total_in_split = len(df_full)

    has_asp = df_full["aspects"].apply(
        lambda x: len(x) > 0 if isinstance(x, (list, np.ndarray)) else False
    )
    no_aspect_count = int((~has_asp).sum())
    df = df_full[has_asp].reset_index(drop=True)
    del df_full
    gc.collect()

    n_chunks = math.ceil(len(df) / CHUNK_SIZE)

    # Validate checkpoint
    if ckpt.get("chunk_size") and ckpt["chunk_size"] != CHUNK_SIZE:
        print(f"WARNING: CHUNK_SIZE thay doi ({ckpt['chunk_size']} -> {CHUNK_SIZE}), reset checkpoint")
        ckpt = {}

    completed = set(ckpt.get("completed_chunks", []))

    print(f"Total rows       : {total_in_split:,}")
    print(f"With aspects     : {len(df):,}")
    print(f"Without aspects  : {no_aspect_count:,}")
    print(f"Chunks           : {n_chunks} (size={CHUNK_SIZE:,})")
    if completed:
        print(f"Checkpoint       : {len(completed)}/{n_chunks} chunks da xong")

Reading: /content/drive/MyDrive/data_p4_kpdl/splits/split_00.parquet
Total rows       : 9,601,840
With aspects     : 4,814,876
Without aspects  : 4,786,964
Chunks           : 97 (size=50,000)


## Load model + Inference

In [ ]:
if ckpt.get("merge_complete"):
    print("Skip — da hoan thanh.")
else:
    remaining = n_chunks - len(completed)

    if remaining > 0:
        print(f"Loading model: {MODEL_PATH}")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device).eval()
        print("Model loaded.\n")

        n_high_total = ckpt.get("n_high", 0)
        n_low_total  = ckpt.get("n_low", 0)
        t_start = time.time()

        pbar = tqdm(total=n_chunks, desc=f"Split {SPLIT_ID}", initial=len(completed))

        for chunk_idx in range(n_chunks):
            if chunk_idx in completed:
                continue

            start = chunk_idx * CHUNK_SIZE
            end   = min(start + CHUNK_SIZE, len(df))
            chunk_df = df.iloc[start:end].reset_index(drop=True)

            high_rows, low_rows = process_chunk(chunk_df, tokenizer, model)

            if high_rows:
                pd.DataFrame(high_rows).to_parquet(
                    os.path.join(CHUNKS_DIR, f"chunk_{chunk_idx:04d}__high.parquet"),
                    index=False,
                )
            if low_rows:
                pd.DataFrame(low_rows).to_parquet(
                    os.path.join(CHUNKS_DIR, f"chunk_{chunk_idx:04d}__low.parquet"),
                    index=False,
                )

            n_high_total += len(high_rows)
            n_low_total  += len(low_rows)
            completed.add(chunk_idx)

            save_checkpoint({
                "split_id":         SPLIT_ID,
                "chunk_size":       CHUNK_SIZE,
                "total_chunks":     n_chunks,
                "completed_chunks": sorted(completed),
                "n_high":           n_high_total,
                "n_low":            n_low_total,
            })

            pbar.update(1)

            del chunk_df, high_rows, low_rows
            gc.collect()
            if device == "cuda":
                torch.cuda.empty_cache()

        pbar.close()
        elapsed = time.time() - t_start
        print(f"\nInference done in {elapsed:.0f}s "
              f"({len(df) / max(elapsed, 1):.0f} rows/s)")

        del model, tokenizer
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()
    else:
        print("All chunks da xong. Chuyen sang merge.")

Loading model: /content/drive/MyDrive/absa_self_train_phase1/asc_teacher_phase1


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded.



Split 0:   0%|          | 0/97 [00:00<?, ?it/s]


Inference done in 4503s (1069 rows/s)


## Merge chunks + Save final

In [ ]:
if ckpt.get("merge_complete"):
    print("Skip — da merge xong.")
else:
    print("Merging chunks -> final files...")
    final_high = merge_chunks("high")
    final_low  = merge_chunks("low")

    # Save split stats
    stats = {
        "split_id":            SPLIT_ID,
        "total_rows_in_split": total_in_split,
        "with_aspects":        int(len(df)),
        "no_aspect_count":     no_aspect_count,
        "n_high":              final_high,
        "n_low":               final_low,
    }
    stats_path = os.path.join(OUTPUT_DIR, "split_stats.json")
    with open(stats_path, "w") as f:
        json.dump(stats, f, indent=2)

    # Mark complete
    save_checkpoint({
        "split_id":         SPLIT_ID,
        "chunk_size":       CHUNK_SIZE,
        "total_chunks":     n_chunks,
        "completed_chunks": sorted(completed),
        "n_high":           final_high,
        "n_low":            final_low,
        "merge_complete":   True,
    })

    print(f"Split {SPLIT_ID} HOAN THANH")
    print(f"  High confidence : {final_high:,}")
    print(f"  Low confidence  : {final_low:,}")
    print(f"  Stats           : {stats_path}")

Merging chunks -> final files...
  high: 4,211,757 rows -> /content/drive/MyDrive/data_p4_kpdl/asc_output/split_00/asc_high_kindle_store.parquet
  low: 603,119 rows -> /content/drive/MyDrive/data_p4_kpdl/asc_output/split_00/asc_low_kindle_store.parquet
Split 0 HOAN THANH
  High confidence : 4,211,757
  Low confidence  : 603,119
  Stats           : /content/drive/MyDrive/data_p4_kpdl/asc_output/split_00/split_stats.json


## Kiểm tra kết quả

In [ ]:
high_path = os.path.join(OUTPUT_DIR, f"asc_high_{CATEGORY_OUTPUT}.parquet")
low_path  = os.path.join(OUTPUT_DIR, f"asc_low_{CATEGORY_OUTPUT}.parquet")

if os.path.exists(high_path):
    h = pd.read_parquet(high_path, columns=["sentence_text", "aspects", "sentiments"])
    print(f"High: {len(h):,} rows")
    print(h.head(3).to_string())
    print()

if os.path.exists(low_path):
    l = pd.read_parquet(low_path, columns=["sentence_text", "aspects"])
    print(f"Low: {len(l):,} rows")
    print(l.head(3).to_string())

High: 4,211,757 rows
                                                                                                                                                sentence_text                    aspects                                                                                                                             sentiments
0          as two opposites attract who are so full of life and love fire and passion angst and turmoil and set this smart funny sexy must read story on fire                    [story]                                                                    [[0.004772864282131195, 0.0030906342435628176, 0.9921364784240723]]
1                                                                                                                       their verbal sparring was enthralling          [verbal sparring]                                                                    [[0.003651039209216833, 0.0026271778624504805, 0.9937216639518738]]
2  i truly felt the